softmax 回归本质上就是一个单层的神经网络
训练目标：最大化对数似然

In [2]:
"准备好训练的数据集"
import os
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from d2l import torch as d2l

os.environ["TORCHVISION_DATASET_URL_BASE"] = "https://mirrors.tuna.tsinghua.edu.cn/github-release/zalandoresearch/fashion-mnist/LATEST/data/"
d2l.use_svg_display()

trans = transforms.ToTensor() # 图像转换器
mnist_train = torchvision.datasets.FashionMNIST(root="./TrainingFigure", train=True, transform=trans, download=True)
mnist_test = torchvision.datasets.FashionMNIST(root="./TrainingFigure", train=False, transform=trans, download=True)

100%|██████████| 26.4M/26.4M [27:02<00:00, 16.3kB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 68.9kB/s]
100%|██████████| 4.42M/4.42M [05:24<00:00, 13.6kB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.53MB/s]


In [6]:
def get_fashion_mnist_labels(labels):
    """返回Fashion-MNIST数据集的文本标签"""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat', 'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    result = []
    for i in labels:
        num = int(i) # 把张量格式的数字转换成整数
        name = text_labels[num]
        result.append(name) # 数组元素追加
    return result

In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    """绘制列表"""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        if torch.is_tensor(img):
            ax.imshow(img.numpy())
        else:
            ax.imshow(img)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes


In [8]:
from IPython import display

batch_size = 256

def get_dataloader_workers():
    return 4

def load_data_fashion_mnist(batch_size, resize=None):
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(root="./TrainingFigure", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(root="./TrainingFigure", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=get_dataloader_workers()), data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=get_dataloader_workers()))

train_iter, test_iter = load_data_fashion_mnist(batch_size)

In [13]:
from torch import nn

class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    def __getitem__(self, idx):
        return self.data[idx]

def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

def train_epoch_ch3(net, train_iter, loss, updater):
    metric = Accumulator(3)
    net.train()
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2], metric[1] / metric[2]

def evaluate_accuracy_ch3(net, data_iter):
    if isinstance(net, nn.Module):
        net.eval()
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

# 训练函数
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    records = []  # 存放每一轮指标
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy_ch3(net, test_iter)
        # 保存本轮数据
        records.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_acc": test_acc
        })

    final_loss = records[-1]["train_loss"]
    final_train_acc = records[-1]["train_acc"]
    final_test_acc = records[-1]["test_acc"]
    return records, final_loss, final_train_acc, final_test_acc

In [14]:
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)) # nn.Linear 只能接收一维向量，因此使用 Flatten() 函数将图片展平为向量

# 参数初始化
linear_layer = net[1] # net[0] 是 nn.Flatten()，没有 weight 等参数，net[1] 才有
nn.init.normal_(linear_layer.weight, std=0.01)

# 损失函数
loss = nn.CrossEntropyLoss(reduction='none')

# 优化算法
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

# 训练
num_epochs = 10
records, last_loss, last_train_acc, last_test_acc = train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

In [ ]:
# Predict
def predict(net, test_iter, n=10):
    for X, y in test_iter:
        break
    trues = get_fashion_mnist_labels(y)
    preds = get_fashion_mnist_labels(net(X).argmax(axis=1))
    titles = [true + '\n' + pred for true, pred in zip(trues, preds)]
    show_images(X[0:n].reshape((n, 28, 28)), 1, n, titles=titles[0:n])

predict(net, test_iter, 10)